# Dialogue Summarization: LoRA vs Adapter Layers vs Zero-Shot Prompting

**Model:** `google/flan-t5-large`
**Dataset:** [`knkarthick/samsum`](https://huggingface.co/datasets/knkarthick/samsum) (SAMSum Corpus, Gliwa et al., 2019)
**Compute:** Kaggle Notebook — 2x NVIDIA T4 GPUs

This notebook runs, in order:
1. Environment setup
2. Data loading & exploration
3. Zero-shot prompting baseline
4. One-shot in-context learning
5. Few-shot in-context learning
6. LoRA fine-tuning
7. Adapter-layer fine-tuning
8. Full comparison of all 5 approaches (ROUGE-1/2/L + BERTScore + efficiency)
9. Model selection
10. Export artifacts + Streamlit app for deployment

> **Kaggle setup:** In the notebook settings (right sidebar), set **Accelerator → GPU T4 x2**
> and **Internet → On** (needed to download the model/dataset).

> **License note:** SAMSum is distributed under CC BY-NC-ND 4.0 (research / non-commercial
> use only). Cite: Gliwa, B., Mochol, I., Biesek, M., & Wawer, A. (2019). *SAMSum Corpus:
> A Human-annotated Dialogue Dataset for Abstractive Summarization.*


## 1. Environment Setup

In [1]:
# Install/upgrade required libraries (Kaggle usually has torch + transformers preinstalled,
# but we pin versions known to work together for PEFT + adapters + T5).
!pip install -q -U transformers datasets accelerate evaluate
!pip install -q -U peft
!pip install -q -U adapters          # AdapterHub's modern adapter library (successor to adapter-transformers)
!pip install -q -U rouge_score bert_score
!pip install -q -U sentencepiece
!pip install -q -U py7zr             # harmless if unused; some HF dataset scripts need it
!pip install -q -U torchao
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 93.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.5/295.5 kB 7.8 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 12.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━

In [3]:
import os, time, json, random
import numpy as np
import pandas as pd
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


Torch version: 2.10.0+cu128
CUDA available: True
GPU count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


**Note on multi-GPU (2x T4):** Hugging Face `Trainer` automatically uses all visible
GPUs via `DataParallel` when you don't launch with `accelerate launch` / `torchrun`.
That's sufficient here — no extra multi-GPU code needed, `Trainer` detects
`torch.cuda.device_count() == 2` and splits batches across both GPUs automatically.

## 2. Load & Explore the Dataset

In [4]:
from datasets import load_dataset

raw_datasets = load_dataset("knkarthick/samsum")
print(raw_datasets)

print("\nSample record:")
print(raw_datasets["train"][0])


README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

Sample record:
{'id': '13818513', 'dialogue': "Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)", 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}


In [5]:
print("\nSample record:")
print(raw_datasets["train"][10])


Sample record:
{'id': '13727633', 'dialogue': 'Lucas: Hey! How was your day?\nDemi: Hey there! \nDemi: It was pretty fine, actually, thank you!\nDemi: I just got promoted! :D\nLucas: Whoa! Great news!\nLucas: Congratulations!\nLucas: Such a success has to be celebrated.\nDemi: I agree! :D\nDemi: Tonight at Death & Co.?\nLucas: Sure!\nLucas: See you there at 10pm?\nDemi: Yeah! See you there! :D', 'summary': 'Demi got promoted. She will celebrate that with Lucas at Death & Co at 10 pm.'}


In [6]:
# Quick sanity stats
for split in raw_datasets:
    dialogue_lens = [len(x.split()) for x in raw_datasets[split]["dialogue"]]
    summary_lens = [len(x.split()) for x in raw_datasets[split]["summary"]]
    print(f"{split}: n={len(raw_datasets[split])}, "
          f"avg dialogue words={np.mean(dialogue_lens):.1f}, "
          f"avg summary words={np.mean(summary_lens):.1f}")


train: n=14731, avg dialogue words=93.8, avg summary words=20.3
validation: n=818, avg dialogue words=91.6, avg summary words=20.3
test: n=819, avg dialogue words=95.5, avg summary words=20.0


In [7]:
# For faster iteration during development, you can optionally subsample.
# Set FULL_RUN = True before your final overnight run to use the entire dataset.
FULL_RUN = True

if not FULL_RUN:
    raw_datasets["train"] = raw_datasets["train"].shuffle(seed=SEED).select(range(700))
    raw_datasets["validation"] = raw_datasets["validation"].shuffle(seed=SEED).select(range(70))
    raw_datasets["test"] = raw_datasets["test"].shuffle(seed=SEED).select(range(70))

print({k: len(v) for k, v in raw_datasets.items()})


{'train': 14731, 'validation': 818, 'test': 819}


## 3. Load Base Model, Tokenizer, and Evaluation Metrics

In [28]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
base_model.to(DEVICE)

MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 100


In [9]:
import evaluate

rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

def compute_all_metrics(predictions, references):
    """Returns a dict with ROUGE-1/2/L and BERTScore F1 (averaged)."""
    rouge_result = rouge_metric.compute(
        predictions=predictions, references=references, use_stemmer=True
    )
    bertscore_result = bertscore_metric.compute(
        predictions=predictions, references=references, lang="en"
    )
    return {
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        "bertscore_f1": float(np.mean(bertscore_result["f1"])),
    }

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [10]:
def generate_summary(model, tokenizer, prompt_text, max_new_tokens=100):
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True,
                        max_length=MAX_INPUT_LENGTH).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


## 4. Zero-Shot Prompting Baseline

We prompt the base FLAN-T5-large model with an instruction template and no examples.

In [29]:
ZERO_SHOT_TEMPLATE = "Summarize the following conversation:\n{dialogue}\nSummary:"

def run_zero_shot_eval(model, tokenizer, test_split, n_samples=100, tag="zero_shot"):
    subset = test_split if n_samples is None else test_split.select(range(n_samples))
    preds, refs, latencies = [], [], []
    for ex in subset:
        prompt = ZERO_SHOT_TEMPLATE.format(dialogue=ex["dialogue"])
        t0 = time.time()
        pred = generate_summary(model, tokenizer, prompt)
        latencies.append(time.time() - t0)
        preds.append(pred)
        refs.append(ex["summary"])
    metrics = compute_all_metrics(preds, refs)
    metrics["avg_latency_sec"] = float(np.mean(latencies))
    metrics["method"] = tag
    return metrics, preds, refs

zero_shot_metrics, zero_shot_preds, zero_shot_refs = run_zero_shot_eval(
    base_model, tokenizer, raw_datasets["test"]
)
print(json.dumps(zero_shot_metrics, indent=2))

{
  "rouge1": 0.4965693697373149,
  "rouge2": 0.23926407302878408,
  "rougeL": 0.4080017546907406,
  "bertscore_f1": 0.9139087837934494,
  "avg_latency_sec": 0.5973372483253478,
  "method": "zero_shot"
}


In [30]:
import pandas as pd

comparison_df = pd.DataFrame({
    "prediction": zero_shot_preds,
    "reference": zero_shot_refs
})

pd.set_option("display.max_colwidth", None)  # don't truncate long text
comparison_df

,prediction,reference
0,Amanda can't find Betty's number. Hannah doesn't know him well. Amanda texted Larry last time they were at the park together.,Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
1,Eric likes Rob's stand-ups on youtube.,Eric and Rob are going to watch a stand-up on youtube.
2,Lenny wants to buy two pairs of purple trousers. Bob recommends the first pair or the third pair.,Lenny can't decide which trousers to buy. Bob advised Lenny on that topic. Lenny goes with Bob's advice to pick the trousers that are of best quality.
3,Emma will be home soon. Will will pick her up when she gets home.,Emma will be home soon and she will let Will know.
4,Jane lost her calendar. Ollie and Jane have lunch on Friday. Jane will be in Morocco at 6 pm. Ollie will bring pastries.,Jane is in Warsaw. Ollie and Jane has a party. Jane lost her calendar. They will get a lunch this week on Friday. Ollie accidentally called Jane and talked about whisky. Jane cancels lunch. They'll meet for a tea at 6 pm.
...,...,...
95,"Flo can't go to the salon until the 6th, because she's going to be gray. Gina will get her a touch-up kit at Tesco.",Flo cannot get an appointment at the salon until the 6th. Flo worries she's going to be gray. Flo will have to get a touch-up kit at Tesco.
96,Ann can't pick up Rob's phone because he's at the grocery store. Rob will pick up Ann's phone.,"Rob is doing shopping at the grocery store. Ann ordered him to buy a cucumber, some tomatoes, bananas and apples."
97,Melany doesn't remember when she got laid.,It's been very long since Melany last had sex. Marvin made an inappropriate joke about it.
98,Noah and Noah's favorite professor is talking about this recent scandal on the news.,"Eric, Samantha and Noah's professor is commenting a recent scandal on the news."


In [31]:
for pred, ref in zip(zero_shot_preds, zero_shot_refs):
    print(f"{'PRED:':<8}{pred}")
    print(f"{'REF:':<8}{ref}")
    print("-" * 80)

PRED:   Amanda can't find Betty's number. Hannah doesn't know him well. Amanda texted Larry last time they were at the park together.
REF:    Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
--------------------------------------------------------------------------------
PRED:   Eric likes Rob's stand-ups on youtube.
REF:    Eric and Rob are going to watch a stand-up on youtube.
--------------------------------------------------------------------------------
PRED:   Lenny wants to buy two pairs of purple trousers. Bob recommends the first pair or the third pair.
REF:    Lenny can't decide which trousers to buy. Bob advised Lenny on that topic. Lenny goes with Bob's advice to pick the trousers that are of best quality.
--------------------------------------------------------------------------------
PRED:   Emma will be home soon. Will will pick her up when she gets home.
REF:    Emma will be home soon and she will let Will know.
-----------------------

## 5. In-Context Learning: One-Shot and Few-Shot

We prepend 1 (one-shot) or several (few-shot) example dialogue→summary pairs
before the target dialogue, then ask the model to summarize the target.
Note: encoder-decoder models like FLAN-T5 have a smaller context window than
large decoder-only LLMs, so few-shot examples are kept short.

In [32]:
def build_icl_prompt(example_pool, target_dialogue, k):
    """Builds a k-shot prompt using k examples drawn from the training set."""
    examples = example_pool.shuffle(seed=SEED).select(range(k))
    prompt_parts = []
    for ex in examples:
        prompt_parts.append(
            f"Conversation:\n{ex['dialogue']}\nSummary: {ex['summary']}\n"
        )
    prompt_parts.append(f"Conversation:\n{target_dialogue}\nSummary:")
    return "\n".join(prompt_parts)


def run_icl_eval(model, tokenizer, train_split, test_split, k, n_samples=100, tag="icl"):
    subset = test_split if n_samples is None else test_split.select(range(n_samples))
    preds, refs, latencies = [], [], []
    for ex in subset:
        prompt = build_icl_prompt(train_split, ex["dialogue"], k)
        t0 = time.time()
        pred = generate_summary(model, tokenizer, prompt)
        latencies.append(time.time() - t0)
        preds.append(pred)
        refs.append(ex["summary"])
    metrics = compute_all_metrics(preds, refs)
    metrics["avg_latency_sec"] = float(np.mean(latencies))
    metrics["method"] = tag
    return metrics, preds, refs

# One-shot (k=1)
one_shot_metrics, one_shot_preds, one_shot_refs = run_icl_eval(
    base_model, tokenizer, raw_datasets["train"], raw_datasets["test"], k=1, tag="one_shot"
)
print("One-shot:", json.dumps(one_shot_metrics, indent=2))

# Few-shot (k=4)
few_shot_metrics, few_shot_preds, few_shot_refs = run_icl_eval(
    base_model, tokenizer, raw_datasets["train"], raw_datasets["test"], k=4, tag="few_shot"
)
print("Few-shot:", json.dumps(few_shot_metrics, indent=2))


One-shot: {
  "rouge1": 0.4945631728075681,
  "rouge2": 0.23041226276435853,
  "rougeL": 0.4038844662155187,
  "bertscore_f1": 0.9135992950201035,
  "avg_latency_sec": 0.635456063747406,
  "method": "one_shot"
}
Few-shot: {
  "rouge1": 0.2073441119878621,
  "rouge2": 0.0501316928299145,
  "rougeL": 0.18093176293837565,
  "bertscore_f1": 0.8759406781196595,
  "avg_latency_sec": 0.4521192955970764,
  "method": "few_shot"
}


In [33]:
import pandas as pd

# Build dataframes
one_shot_df = pd.DataFrame({"prediction": one_shot_preds, "reference": one_shot_refs})
few_shot_df = pd.DataFrame({"prediction": few_shot_preds, "reference": few_shot_refs})

# Save them too, so you have a record without re-running generation
one_shot_df.to_csv(f"{OUTPUT_DIR}/one_shot_preds.csv", index=False)
few_shot_df.to_csv(f"{OUTPUT_DIR}/few_shot_preds.csv", index=False)

In [34]:
def print_pred_ref_pairs(df, n=None, label=""):
    subset = df if n is None else df.head(n)
    if label:
        print(f"=== {label} ===\n")
    for pred, ref in zip(subset["prediction"], subset["reference"]):
        print(f"{'PRED:':<8}{pred}")
        print(f"{'REF:':<8}{ref}")
        print("-" * 80)

In [35]:
print_pred_ref_pairs(one_shot_df, n=10, label="One-Shot")
print_pred_ref_pairs(few_shot_df, n=10, label="Few-Shot")

=== One-Shot ===

PRED:   Amanda can't find Betty's number. Hannah doesn't know him well. Amanda texted Betty last time they were at the park together.
REF:    Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
--------------------------------------------------------------------------------
PRED:   Eric likes Rob's stand-ups on youtube.
REF:    Eric and Rob are going to watch a stand-up on youtube.
--------------------------------------------------------------------------------
PRED:   Lenny wants to buy two pairs of purple trousers. Bob recommends the first pair or the third pair.
REF:    Lenny can't decide which trousers to buy. Bob advised Lenny on that topic. Lenny goes with Bob's advice to pick the trousers that are of best quality.
--------------------------------------------------------------------------------
PRED:   Emma will be home soon. Will will pick her up when she gets home.
REF:    Emma will be home soon and she will let Will know.
-----

## 6. Preprocessing for Fine-Tuning (shared by LoRA and Adapter runs)

In [11]:
FT_PROMPT_PREFIX = "Summarize the following conversation:\n"

def preprocess_function(examples):
    inputs = [FT_PROMPT_PREFIX + d for d in examples["dialogue"]]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=examples["summary"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(
    preprocess_function, batched=True,
    remove_columns=raw_datasets["train"].column_names
)
print(tokenized_datasets)


Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})


## 7. LoRA Fine-Tuning

We freeze the base FLAN-T5-large weights and inject low-rank adapters
into the attention projection layers using Hugging Face `peft`.

In [10]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import DataCollatorForSeq2Seq
import warnings
warnings.filterwarnings("ignore")
lora_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q", "v"],   # T5 attention projection layers
    bias="none",
)

lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()
lora_model.to(DEVICE)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=lora_model, padding=True
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 1,769,472 || all params: 249,347,328 || trainable%: 0.7096


In [12]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

lora_training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-samsum-lora",
    eval_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=4, 
    per_device_eval_batch_size=4,
    num_train_epochs=5, 
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=True,     
    logging_strategy="steps",
    logging_steps=50,
    optim="paged_adamw_8bit", # Efficient optimizer for memory constraints
    report_to="none",
    generation_max_length=MAX_TARGET_LENGTH,
)


lora_trainer = Seq2SeqTrainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

lora_train_start = time.time()
lora_trainer.train()
lora_train_time = time.time() - lora_train_start
print(f"LoRA training time: {lora_train_time/60:.1f} minutes")

lora_trainer.save_model(f"{OUTPUT_DIR}/lora_flan_t5_large_final")

Epoch,Training Loss,Validation Loss
1,1.465300,1.393940
2,1.456800,1.381410
3,1.369500,1.374637
4,1.342600,1.376209
5,1.358300,1.374077


LoRA training time: 95.0 minutes


In [13]:
import os
import shutil
from IPython.display import FileLink

# Path to Kaggle's output directory
output_dir = '/kaggle/working'

# Create a zip archive of everything in the output directory
zip_path = '/kaggle/working/all_output'
shutil.make_archive(zip_path, 'zip', output_dir)

# Generate a downloadable link (works in Kaggle notebooks)
FileLink(r'all_output.zip')

/kaggle/working/all_output.zip

In [20]:
def run_finetuned_eval(model, tokenizer, test_split, n_samples=10, tag="finetuned"):
    subset = test_split if n_samples is None else test_split.select(range(n_samples))
    preds, refs, latencies = [], [], []
    for ex in subset:
        prompt = FT_PROMPT_PREFIX + ex["dialogue"]
        t0 = time.time()
        pred = generate_summary(model, tokenizer, prompt)
        latencies.append(time.time() - t0)
        preds.append(pred)
        refs.append(ex["summary"])
    metrics = compute_all_metrics(preds, refs)
    metrics["avg_latency_sec"] = float(np.mean(latencies))
    metrics["method"] = tag
    return metrics, preds, refs

In [14]:

lora_metrics, lora_preds, lora_refs = run_finetuned_eval(
    lora_model, tokenizer, raw_datasets["test"], tag="lora"
)
lora_trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
lora_total_params = sum(p.numel() for p in lora_model.parameters())
lora_metrics["trainable_params_pct"] = 100 * lora_trainable_params / lora_total_params
lora_metrics["train_time_min"] = lora_train_time / 60
print(json.dumps(lora_metrics, indent=2))

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{
  "rouge1": 0.4952010521777964,
  "rouge2": 0.23035459156472995,
  "rougeL": 0.3639271780229228,
  "bertscore_f1": 0.91534503698349,
  "avg_latency_sec": 1.007106328010559,
  "method": "lora",
  "trainable_params_pct": 0.7096414524241463,
  "train_time_min": 94.99555478493373
}


In [15]:
import pandas as pd

comparison_df = pd.DataFrame({
    "prediction": lora_preds,
    "reference": lora_refs
})

pd.set_option("display.max_colwidth", None)  # don't truncate long text
comparison_df

,prediction,reference
0,Amanda can't find Betty's number. Amanda will ask Larry. Larry called Betty last time they were at the park together. Amanda will text Larry.,Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
1,Eric and Rob like Eric's stand-ups on youtube.,Eric and Rob are going to watch a stand-up on youtube.
2,Lenny wants to buy two pairs of purple trousers. Bob has four black trousers.,Lenny can't decide which trousers to buy. Bob advised Lenny on that topic. Lenny goes with Bob's advice to pick the trousers that are of best quality.
3,Emma will be home soon and will tell Will when she gets home.,Emma will be home soon and she will let Will know.
4,Jane lost her calendar. Ollie and Jane have lunch this week. They have lunch on Friday. Jane will be in Morocco at 6 pm. Ollie will bring some sun with her. Jane will bring pastries.,Jane is in Warsaw. Ollie and Jane has a party. Jane lost her calendar. They will get a lunch this week on Friday. Ollie accidentally called Jane and talked about whisky. Jane cancels lunch. They'll meet for a tea at 6 pm.
5,Hilary and Elliot are meeting at the entrance to the conference hall at 2 pm and going to La Cantina. They will have lunch with French people who work on the history of food in colonial Mexico.,Hilary has the keys to the apartment. Benjamin wants to get them and go take a nap. Hilary is having lunch with some French people at La Cantina. Hilary is meeting them at the entrance to the conference hall at 2 pm. Benjamin and Elliot might join them. They're meeting for the drinks in the evening.
6,"Payton buys clothes from 2 or 3 of the websites Max recommends. Max will check them out. Payton likes browsing, trying on, looking in the mirror and seeing how he looks. He also likes books.",Payton provides Max with websites selling clothes. Payton likes browsing and trying on the clothes but not necessarily buying them. Payton usually buys clothes and books as he loves reading.
7,Rita is tired at work. Tina hates her job.,Rita and Tina are bored at work and have still 4 hours left.
8,"Beatrice is in town shopping. They have nice scarfs in the shop next to the church. Leo doesn't want one, because he has a cold all the time. Beatrice will buy him a scarf.","Beatrice wants to buy Leo a scarf, but he doesn't like scarves. She cares about his health and will buy him a scarf no matter his opinion."
9,Eric is coming to Ivan's brother's wedding. He has a lot to do at home and doesn't know if his parents will let him. Ivan will take care of Eric's parents.,Eric doesn't know if his parents let him go to Ivan's brother's wedding. Ivan will talk to them.


In [16]:
for pred, ref in zip(lora_preds, lora_refs):
    print(f"{'PRED:':<8}{pred}")
    print(f"{'REF:':<8}{ref}")
    print("-" * 80)

PRED:   Amanda can't find Betty's number. Amanda will ask Larry. Larry called Betty last time they were at the park together. Amanda will text Larry.
REF:    Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
--------------------------------------------------------------------------------
PRED:   Eric and Rob like Eric's stand-ups on youtube.
REF:    Eric and Rob are going to watch a stand-up on youtube.
--------------------------------------------------------------------------------
PRED:   Lenny wants to buy two pairs of purple trousers. Bob has four black trousers.
REF:    Lenny can't decide which trousers to buy. Bob advised Lenny on that topic. Lenny goes with Bob's advice to pick the trousers that are of best quality.
--------------------------------------------------------------------------------
PRED:   Emma will be home soon and will tell Will when she gets home.
REF:    Emma will be home soon and she will let Will know.
-----------------------

### 7.2. Inference on Unseen Text

In [23]:
unseen_text = """
Summarize the following conversation:
Sara: Hey Ali, have you finished the machine learning assignment?
Ali: Not yet. I completed the data preprocessing part, but I'm still working on the model evaluation.
Sara: The submission deadline is tomorrow at 5 PM.
Ali: I know. I plan to finish it tonight and double-check the results.
Sara: Great. Don't forget to include the confusion matrix and ROC curve in your report.
Ali: Thanks for reminding me. I'll also upload the notebook to GitHub before submitting.
Sara: Sounds good. Let me know if you need any help.
Ali: Will do. Thanks!
"""

inputs = tokenizer(unseen_text, return_tensors="pt").to(lora_model.device)

outputs = lora_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],   # good practice to pass this too
    max_new_tokens=70,
)
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nGenerated Summary:\n{summary}")


Generated Summary:
Ali hasn't finished the machine learning assignment yet. He will finish the model evaluation tonight and double-check the results. Sara reminds him to include the confusion matrix and ROC curve in his report. Ali will upload the notebook to GitHub before submitting.


## 8. Adapter-Layer Fine-Tuning

Same setup as LoRA, but instead of low-rank matrices we insert bottleneck
adapter modules using the AdapterHub `adapters` library, and train only
those adapter parameters (base weights stay frozen).

In [15]:
import adapters
from adapters import SeqBnConfig

adapter_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
adapters.init(adapter_model)

ADAPTER_NAME = "samsum_summarization_adapter"
adapter_config = SeqBnConfig(reduction_factor=16)

adapter_model.add_adapter(ADAPTER_NAME, config=adapter_config)
adapter_model.train_adapter(ADAPTER_NAME)
adapter_model.set_active_adapters(ADAPTER_NAME)
adapter_model.to(DEVICE)

trainable = sum(p.numel() for p in adapter_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in adapter_model.parameters())
print(f"Adapter trainable params: {trainable:,} ({100*trainable/total:.3f}% of total)")


There are adapters available but none are activated for the forward pass.


Adapter trainable params: 1,789,056 (0.717% of total)


In [16]:
from transformers import Seq2SeqTrainer

class AdapterCompatibleTrainer(Seq2SeqTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # The adapters library's T5 wrapper doesn't accept num_items_in_batch,
        # so we simply don't pass it through.
        return super().compute_loss(model, inputs, return_outputs=return_outputs)

In [17]:
# Fresh collator bound to the adapter model, not the LoRA one
from transformers import DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

adapter_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=adapter_model, padding=True
)

adapter_training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-samsum-adapter",
    eval_strategy="epoch",
    learning_rate=5e-4,                    # matched to LoRA's LR for fair comparison
    per_device_train_batch_size=4,          # matched
    per_device_eval_batch_size=4,           # matched
    num_train_epochs=5,                     # matched
    weight_decay=0.01,                      # matched
    save_total_limit=2,                     # matched
    predict_with_generate=True,             # matched
    fp16=True,                              # matched
    logging_strategy="steps",
    logging_steps=50,                       # matched
    optim="paged_adamw_8bit",               # matched
    report_to="none",
    generation_max_length=MAX_TARGET_LENGTH, # matched
)

adapter_trainer = AdapterCompatibleTrainer(
    model=adapter_model,
    args=adapter_training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=adapter_data_collator,     # fixed: uses adapter_data_collator, not the LoRA one
    tokenizer=tokenizer,
)

adapter_train_start = time.time()
adapter_trainer.train()
adapter_train_time = time.time() - adapter_train_start
print(f"Adapter training time: {adapter_train_time/60:.1f} minutes")

adapter_model.save_adapter(f"{OUTPUT_DIR}/adapter_flan_t5_large_final", ADAPTER_NAME)

/tmp/ipykernel_58/3423762444.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `AdapterCompatibleTrainer.__init__`. Use `processing_class` instead.
  adapter_trainer = AdapterCompatibleTrainer(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1830: FutureWarning: `past_key_value` is deprecated and will be removed in version 4.58 for `T5Block.forward`. Use `past_key_values` instead.
  result = forward_call(*args, **kwargs)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
past_key_values should not be None in from_legacy_cache()
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().a

Epoch,Training Loss,Validation Loss
1,1.473900,1.389051
2,1.463700,1.380885
3,1.377000,1.379278
4,1.337500,1.376835
5,1.351000,1.378253


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1830: FutureWarning: `past_key_value` is deprecated and will be removed in version 4.58 for `T5Block.forward`. Use `past_key_values` instead.
  result = forward_call(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1830: FutureWarning: `past_key_value` is deprecated and will be removed in version 4.58 for `T5Block.forward`. Use `past_key_values` instead.
  result = forward_call(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, 

Adapter training time: 89.7 minutes


In [18]:
import os
import shutil
from IPython.display import FileLink

# Path to Kaggle's output directory
output_dir = '/kaggle/working'

# Create a zip archive of everything in the output directory
zip_path = '/kaggle/working/all_output'
shutil.make_archive(zip_path, 'zip', output_dir)

# Generate a downloadable link (works in Kaggle notebooks)
FileLink(r'all_output.zip')

/kaggle/working/all_output.zip

In [21]:
adapter_metrics, adapter_preds, adapter_refs = run_finetuned_eval(
    adapter_model, tokenizer, raw_datasets["test"], tag="adapter"
)
adapter_trainable_params = sum(p.numel() for p in adapter_model.parameters() if p.requires_grad)
adapter_total_params = sum(p.numel() for p in adapter_model.parameters())
adapter_metrics["trainable_params_pct"] = 100 * adapter_trainable_params / adapter_total_params
adapter_metrics["train_time_min"] = adapter_train_time / 60
print(json.dumps(adapter_metrics, indent=2))


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{
  "rouge1": 0.4594095118822472,
  "rouge2": 0.19923974395501134,
  "rougeL": 0.3587746976588856,
  "bertscore_f1": 0.9112975478172303,
  "avg_latency_sec": 1.069811224937439,
  "method": "adapter",
  "trainable_params_pct": 0.7174392086148141,
  "train_time_min": 89.707670434316
}


In [22]:
import pandas as pd

comparison_df = pd.DataFrame({
    "prediction": adapter_preds,
    "reference": adapter_refs
})

pd.set_option("display.max_colwidth", None)  # don't truncate long text
comparison_df

,prediction,reference
0,Amanda can't find Betty's number. Amanda asks Larry to call her last time they were at the park together.,Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
1,Eric and Rob like Eric's stand-ups on youtube.,Eric and Rob are going to watch a stand-up on youtube.
2,Lenny will buy the first or the third pair of trousers.,Lenny can't decide which trousers to buy. Bob advised Lenny on that topic. Lenny goes with Bob's advice to pick the trousers that are of best quality.
3,Emma will be home soon. Will will pick her up.,Emma will be home soon and she will let Will know.
4,Ollie and Jane are in Warsaw. They have lunch this week on Friday. Jane will be in Morocco at 6 pm after her courses. Ollie will bring pastries for tea.,Jane is in Warsaw. Ollie and Jane has a party. Jane lost her calendar. They will get a lunch this week on Friday. Ollie accidentally called Jane and talked about whisky. Jane cancels lunch. They'll meet for a tea at 6 pm.
5,Hilary and Elliot are meeting for drinks in the evening. Hilary is meeting French people at the entrance to the conference hall at 2 pm and they will go to La Cantina.,Hilary has the keys to the apartment. Benjamin wants to get them and go take a nap. Hilary is having lunch with some French people at La Cantina. Hilary is meeting them at the entrance to the conference hall at 2 pm. Benjamin and Elliot might join them. They're meeting for the drinks in the evening.
6,Payton buys clothes from 2 or 3 websites. Max will check them out.,Payton provides Max with websites selling clothes. Payton likes browsing and trying on the clothes but not necessarily buying them. Payton usually buys clothes and books as he loves reading.
7,Rita is tired at work. Tina and Rita hate their jobs.,Rita and Tina are bored at work and have still 4 hours left.
8,"Beatrice is in town, shopping. They have nice scarfs in the shop next to the church. Leo doesn't need one, because he had a cold all the time. Leo will get a scarf.","Beatrice wants to buy Leo a scarf, but he doesn't like scarves. She cares about his health and will buy him a scarf no matter his opinion."
9,Eric is coming to Eric's brother's wedding. Ivan will take care of Eric's parents.,Eric doesn't know if his parents let him go to Ivan's brother's wedding. Ivan will talk to them.


In [23]:
for pred, ref in zip(adapter_preds, adapter_refs):
    print(f"{'PRED:':<8}{pred}")
    print(f"{'REF:':<8}{ref}")
    print("-" * 80)

PRED:   Amanda can't find Betty's number. Amanda asks Larry to call her last time they were at the park together.
REF:    Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
--------------------------------------------------------------------------------
PRED:   Eric and Rob like Eric's stand-ups on youtube.
REF:    Eric and Rob are going to watch a stand-up on youtube.
--------------------------------------------------------------------------------
PRED:   Lenny will buy the first or the third pair of trousers.
REF:    Lenny can't decide which trousers to buy. Bob advised Lenny on that topic. Lenny goes with Bob's advice to pick the trousers that are of best quality.
--------------------------------------------------------------------------------
PRED:   Emma will be home soon. Will will pick her up.
REF:    Emma will be home soon and she will let Will know.
--------------------------------------------------------------------------------
PRED:   Ollie a

In [25]:
unseen_text = """
Summarize the following conversation:
Sara: Hey Ali, have you finished the machine learning assignment?
Ali: Not yet. I completed the data preprocessing part, but I'm still working on the model evaluation.
Sara: The submission deadline is tomorrow at 5 PM.
Ali: I know. I plan to finish it tonight and double-check the results.
Sara: Great. Don't forget to include the confusion matrix and ROC curve in your report.
Ali: Thanks for reminding me. I'll also upload the notebook to GitHub before submitting.
Sara: Sounds good. Let me know if you need any help.
Ali: Will do. Thanks!
"""

inputs = tokenizer(unseen_text, return_tensors="pt").to(adapter_model.device)

outputs = adapter_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],   # good practice to pass this too
    max_new_tokens=70,
)
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nGenerated Summary:\n{summary}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1830: FutureWarning: `past_key_value` is deprecated and will be removed in version 4.58 for `T5Block.forward`. Use `past_key_values` instead.
  result = forward_call(*args, **kwargs)



Generated Summary:
Ali hasn't finished the machine learning assignment yet. He will finish the model evaluation tonight and double-check the results. Sara reminds Ali to include the confusion matrix and ROC curve in his report. Ali will upload the notebook to GitHub before submitting the assignment.


## 9. Full Comparison: Zero-Shot vs One-Shot vs Few-Shot vs LoRA vs Adapter

In [37]:
lora_metrics = {
  "rouge1": 0.4952010521777964,
  "rouge2": 0.23035459156472995,
  "rougeL": 0.3639271780229228,
  "bertscore_f1": 0.91534503698349,
  "avg_latency_sec": 1.007106328010559,
  "method": "lora",
  "trainable_params_pct": 0.7096414524241463,
  "train_time_min": 94.99555478493373
}
all_metrics = [
    zero_shot_metrics,
    one_shot_metrics,
    few_shot_metrics,
    lora_metrics,
    adapter_metrics,
]

comparison_df = pd.DataFrame(all_metrics)
comparison_df = comparison_df[[
    "method", "rouge1", "rouge2", "rougeL", "bertscore_f1",
    "avg_latency_sec", "trainable_params_pct", "train_time_min"
]]
comparison_df = comparison_df.round(4)
comparison_df.to_csv(f"{OUTPUT_DIR}/comparison_results.csv", index=False)
comparison_df


,method,rouge1,rouge2,rougeL,bertscore_f1,avg_latency_sec,trainable_params_pct,train_time_min
0,zero_shot,0.4966,0.2393,0.4080,0.9139,0.5973,NaN,NaN
1,one_shot,0.4946,0.2304,0.4039,0.9136,0.6355,NaN,NaN
2,few_shot,0.2073,0.0501,0.1809,0.8759,0.4521,NaN,NaN
3,lora,0.4952,0.2304,0.3639,0.9153,1.0071,0.7096,94.9956
4,adapter,0.4594,0.1992,0.3588,0.9113,1.0698,0.7174,89.7077


### Reading the table

- **ROUGE-1/2/L** — lexical overlap with reference summaries; higher is better.
- **BERTScore F1** — semantic similarity, catches valid paraphrases ROUGE misses.
- **trainable_params_pct** — only applies meaningfully to LoRA/Adapter (0 for prompting methods).
- **avg_latency_sec** — inference time per sample; matters for production deployment.
- **train_time_min** — wall-clock training time (only applies to LoRA/Adapter).

## 10. Model Selection

In [39]:
finetuned_only = comparison_df[comparison_df["method"].isin(["lora", "adapter"])]
winner_row = finetuned_only.sort_values("rougeL", ascending=False).iloc[0]
winner_method = winner_row["method"]

print("Fine-tuned comparison:")
print(finetuned_only)
print(f"\nSelected method: {winner_method.upper()}")
print(f"ROUGE-L: {winner_row['rougeL']:.4f} | BERTScore F1: {winner_row['bertscore_f1']:.4f} | "
      f"Trainable params: {winner_row['trainable_params_pct']:.3f}% | "
      f"Train time: {winner_row['train_time_min']:.1f} min")


Fine-tuned comparison:
    method  rouge1  rouge2  rougeL  bertscore_f1  avg_latency_sec  \
3     lora  0.4952  0.2304  0.3639        0.9153           1.0071   
4  adapter  0.4594  0.1992  0.3588        0.9113           1.0698   

   trainable_params_pct  train_time_min  
3                0.7096         94.9956  
4                0.7174         89.7077  

Selected method: LORA
ROUGE-L: 0.3639 | BERTScore F1: 0.9153 | Trainable params: 0.710% | Train time: 95.0 min


LoRA fine-tuning outperformed Adapter-layer fine-tuning across all four metrics at matched trainable-parameter cost (0.71% vs 0.72%) and similar training time. Neither fine-tuning method improved on FLAN-T5-large's zero-shot baseline, which was unusually strong on this dataset — likely due to FLAN-T5's instruction-tuning already covering summarization-style tasks. This suggests limited additional value from fine-tuning this specific base model on SAMSum, though LoRA remains the better choice between the two PEFT methods if fine-tuning is required."

## 11. Wrap-Up Checklist

- [x] Zero-shot, one-shot, few-shot prompting evaluated
- [x] LoRA fine-tuning + evaluation
- [x] Adapter fine-tuning + evaluation
- [x] ROUGE-1/2/L and BERTScore computed for every method
- [x] Trainable-parameter %, train time, and inference latency logged
- [x] Comparison table saved to `comparison_results.csv`
- [x] Winning model exported for deployment
